> Konstantinos Mpouros <br>
> Github: https://github.com/konstantinosmpouros?tab=repositories<br>
> Year: 2025

This notebook is dedicated to prototyping and testing **Deep Agents** in **mAgenticX**, with emphasis on sub-agent orchestration, streaming behavior, and AG-UI event normalization.


## Libraries & Modules

In [1]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

import uuid

In [2]:
import json
import time
# from uuid import uuid4
# from dataclasses import dataclass
from typing import Any, Dict, List, Literal, Optional, Sequence, Tuple, Union
from pydantic import BaseModel

## AGUI Stream Normalizer & Helper Functions

In [3]:
async def run_stream(user_text: str, stream_mode: str, deep_agent):
    configurable = {"configurable": {"thread_id": str(uuid.uuid4())}}
    inputs = {"messages": [{"role": "user", "content": user_text}]}

    async for chunk in deep_agent.astream(
        inputs,
        stream_mode=stream_mode,
        config=configurable,
    ):
        print(chunk)

In [4]:
def _unwrap_envelope(chunk: Any) -> Tuple[Optional[tuple], Optional[str], Any, Optional[Dict[str, Any]]]:
    """
    For stream_mode=["messages","updates"] with subgraphs=True possible.

    General rule:
    - Find the first string in the chunk that matches an allowed mode.
    - The item just before it (if any) is namespace.
    - The item just after it is payload.
    - The next item (if any) is metadata.
    Legacy: if messages payload is a 2-tuple (msg, meta) and meta is a dict,
    treat meta as metadata when none was provided.
    Fallback: dict chunks => updates; everything else => unknown.
    """
    _ALLOWED_MODES = {"messages", "updates"}

    namespace: Optional[tuple] = None
    mode: Optional[str] = None
    payload: Any = chunk
    metadata: Optional[Dict[str, Any]] = None

    # Sequence parsing: scan for mode string and derive neighbors
    if isinstance(chunk, (tuple, list)):
        seq = list(chunk)
        for idx, item in enumerate(seq):
            if isinstance(item, str) and item in _ALLOWED_MODES:
                mode = item

                # Namespace: immediately before the mode (if present)
                if idx - 1 >= 0:
                    ns_candidate = seq[idx - 1]
                    if isinstance(ns_candidate, tuple) and len(ns_candidate) > 0:
                        namespace = ns_candidate
                    elif isinstance(ns_candidate, list) and len(ns_candidate) > 0:
                        namespace = tuple(ns_candidate)

                # Payload: immediately after the mode (if present)
                if idx + 1 < len(seq):
                    payload = seq[idx + 1]

                # Metadata: the element after payload (if present)
                if idx + 2 < len(seq):
                    meta_candidate = seq[idx + 2]
                    if isinstance(meta_candidate, dict):
                        metadata = meta_candidate

                # Legacy: messages payload can be (msg, meta)
                if (
                    mode == "messages"
                    and isinstance(payload, tuple)
                    and len(payload) == 2
                    and metadata is None
                ):
                    msg, meta = payload
                    payload = msg
                    metadata = meta if isinstance(meta, dict) else None

                return namespace, mode, payload, metadata

    # Fallbacks
    if isinstance(chunk, dict):
        return None, "updates", chunk, None

    return None, None, chunk, None



In [5]:
async def run_stream_unwrap(user_text: str, stream_mode: str, deep_agent):
    configurable = {"configurable": {"thread_id": str(uuid.uuid4())}}
    inputs = {"messages": [{"role": "user", "content": user_text}]}

    async for chunk in deep_agent.astream(
        inputs,
        stream_mode=stream_mode,
        config=configurable,
    ):
        namespace, mode, payload, metadata = _unwrap_envelope(chunk)
        print(f"Namespace: {namespace}")
        print(f"Mode: {mode}")
        print(f"Payload: {payload}")
        print(f"Metadata: {metadata}")

In [6]:
# Custom event names/constants for AG-UI
HITL_INTERRUPT_EVENT_TYPE = "HITL_INTERRUPT"
PLAN_SNAPSHOT_EVENT_TYPE = "PLAN_SNAPSHOT"
TASK_SUBAGENT_EVENT_TYPE = "TASK_SUBAGENT"
SUBAGENT_EVENT_TYPE = "SUBAGENT_EVENT"
BEFORE_AGENT_EVENT_TYPE = "BEFORE_AGENT_EVENT"


# ------------------------------------------------------------------
# HITL Interrupt Event
# ------------------------------------------------------------------
class HITLInterruptEvent(BaseModel):
    """Human-in-the-loop interrupt payload streamed to AG-UI."""
    thread_id: str
    interrupt: Any
    metadata: Optional[Dict[str, Any]] = None


# ------------------------------------------------------------------
# Planning Snapshot Event
# ------------------------------------------------------------------
class PlanItem(BaseModel):
    """Single planning step in a snapshot."""
    content: str
    status: Literal["pending", "in_progress", "completed"]
    metadata: Optional[Dict[str, Any]] = None

class PlanSnapshot(BaseModel):
    """Immutable snapshot of the current plan state."""
    items: List[PlanItem]
    updated_at: Optional[int] = None
    metadata: Optional[Dict[str, Any]] = None


# ------------------------------------------------------------------
# Task -> Sub-agent assignment Event
# ------------------------------------------------------------------
class TaskSubAgentEvent(BaseModel):
    """Describes a task delegated to a sub-agent."""
    task_id: str
    subagent_type: str
    description: str


# ------------------------------------------------------------------
# Sub-agent envelope Event
# ------------------------------------------------------------------
class SubAgentEvent(BaseModel):
    """
    Wraps any normalized AG-UI event emitted by a sub-agent namespace.
    """
    task_id: str
    namespace: List[str]
    event: Dict[str, Any]


# ------------------------------------------------------------------
# Before-agent event
# ------------------------------------------------------------------
class BeforeAgentEvent(BaseModel):
    """
    Captures the delegated instruction observed in
    PatchToolCallsMiddleware.before_agent.
    """
    message: str
    metadata: Optional[Dict[str, Any]] = None



In [7]:
import json
import time
from typing import Any, Dict, Optional, Sequence

from ag_ui.core import (
    EventType,
    
    # General run events
    RunStartedEvent,
    RunFinishedEvent,
    
    # Text message events (assistant responses)
    TextMessageStartEvent,
    TextMessageContentEvent,
    TextMessageEndEvent,
    TextMessageChunkEvent,
    
    # Thinking events
    ThinkingStartEvent,
    ThinkingEndEvent,
    ThinkingTextMessageContentEvent,
    
    # Tool-call events
    ToolCallStartEvent,
    ToolCallArgsEvent,
    ToolCallEndEvent,
    ToolCallResultEvent,
    
    # Safer metrics carrier
    CustomEvent
)
from ag_ui.encoder import EventEncoder



class AGUIEmitter:
    """Stateless AG-UI emitter compatible with LangGraph StreamWriter."""
    def __init__(self) -> None:
        self._encoder = EventEncoder()

    def _emit(self, event_obj: object, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        """Encode and write an event object as SSE bytes, or return bytes when no writer is provided."""
        if getattr(event_obj, "timestamp", None) is None:
            event_obj.timestamp = int(time.time() * 1000)
        sse = self._encoder.encode(event_obj)
        sse = self._attach_namespace(sse, namespace)
        if writer:
            writer(sse)
            return
        return sse

    def _attach_namespace(self, sse: bytes, namespace: Optional[str]) -> bytes:
        """
        Inject a namespace field into the encoded SSE payload.
        Falls back to the original bytes on any parse/encode issue.
        """
        try:
            text = sse.decode("utf-8")
            lines = text.splitlines()

            new_lines = []
            applied = False
            for line in lines:
                if line.startswith("data:"):
                    payload_str = line[len("data:"):].lstrip()
                    payload = json.loads(payload_str)
                    payload["namespace"] = namespace
                    new_lines.append(f"data: {json.dumps(payload, ensure_ascii=False)}")
                    applied = True
                else:
                    new_lines.append(line)

            if not applied:
                return sse

            rebuilt = "\n".join(new_lines)
            if not rebuilt.endswith("\n"):
                rebuilt += "\n"
            if not rebuilt.endswith("\n\n"):
                rebuilt += "\n"

            return rebuilt.encode("utf-8")
        except Exception:
            return sse


    # ---------- Run lifecycle ----------
    def run_start(self, thread_id: str, run_id: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(RunStartedEvent(type=EventType.RUN_STARTED, thread_id=thread_id, run_id=run_id), writer, namespace)

    def run_end(self, thread_id: str, run_id: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(RunFinishedEvent(type=EventType.RUN_FINISHED, thread_id=thread_id, run_id=run_id), writer, namespace)



    # ---------- Thinking session boundaries + content ----------
    def thinking_start(self, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(ThinkingStartEvent(type=EventType.THINKING_START), writer, namespace)

    def thinking_end(self, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(ThinkingEndEvent(type=EventType.THINKING_END), writer, namespace)

    def thought(self, content: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(ThinkingTextMessageContentEvent(type=EventType.THINKING_TEXT_MESSAGE_CONTENT, delta=content), writer, namespace)



    # ---------- Agent message streaming ----------
    def response_start(self, message_id: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(TextMessageStartEvent(type=EventType.TEXT_MESSAGE_START, message_id=message_id), writer, namespace)

    def response_chunk(self, message_id: str, delta: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(TextMessageChunkEvent(type=EventType.TEXT_MESSAGE_CHUNK, message_id=message_id, delta=delta), writer, namespace)

    def response_content(self, message_id: str, delta: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(TextMessageContentEvent(type=EventType.TEXT_MESSAGE_CONTENT, message_id=message_id, delta=delta), writer, namespace)

    def response_end(self, message_id: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(TextMessageEndEvent(type=EventType.TEXT_MESSAGE_END, message_id=message_id), writer, namespace)



    # ---------- Tool calls lifecycle ----------
    def tool_call_start(self, tool_call_id: str, name: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        # Tool Start
        tool_start = ToolCallStartEvent(
            type=EventType.TOOL_CALL_START,
            tool_call_id=tool_call_id,
            tool_call_name=name,
        )
        return self._emit(tool_start, writer, namespace)

    def tool_call_args(self, tool_call_id: str, name: str, args: dict | str | None = None, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        # Tool Args
        tool_args = ToolCallArgsEvent(
            type=EventType.TOOL_CALL_ARGS,
            tool_call_id=tool_call_id,
            delta=json.dumps({"name": name, "args": args or {}}, ensure_ascii=False)
        )
        return self._emit(tool_args, writer, namespace)

    def tool_call_result(self, tool_call_id: str, output: str | dict, writer: Any = None, *, thread_id: Optional[str] = None, namespace: Optional[str] = None) -> Optional[bytes]:
        # Final result wrapper
        message_id = thread_id or tool_call_id
        tool_results = ToolCallResultEvent(
            type=EventType.TOOL_CALL_RESULT,
            tool_call_id=tool_call_id,
            message_id=message_id,
            content=output if isinstance(output, str) else json.dumps(output, ensure_ascii=False),
        )
        return self._emit(tool_results, writer, namespace)

    def tool_call_end(self, tool_call_id: str, writer: Any = None, namespace: Optional[str] = None) -> Optional[bytes]:
        return self._emit(ToolCallEndEvent(type=EventType.TOOL_CALL_END, tool_call_id=tool_call_id,), writer, namespace)



    # ---------- Planning snapshots (custom event) ----------
    def plan_snapshot(
        self,
        items: Sequence[PlanItem | Dict[str, Any]],
        *,
        metadata: Optional[Dict[str, Any]] = None,
        writer: Any = None,
        namespace: Optional[str] = None,
    ) -> CustomEvent:
        """Create a plan snapshot custom event and optionally emit it."""
        snapshot = PlanSnapshot(
            items=list(items),
            updated_at=int(time.time() * 1000),
            metadata=metadata,
        )
        custom_event = CustomEvent(
            type=EventType.CUSTOM,
            name=PLAN_SNAPSHOT_EVENT_TYPE,
            value=snapshot.model_dump(),
        )
        return self._emit(custom_event, writer, namespace)



    # ---------- Task -> sub-agent assignment (custom event) ----------
    def task_subagent(
        self,
        *,
        task_id: str,
        subagent_type: str,
        description: str,
        writer: Any = None,
        namespace: Optional[str] = None,
    ) -> CustomEvent:
        """Create a task->sub-agent custom event and optionally emit it."""
        payload = TaskSubAgentEvent(
            task_id=task_id,
            subagent_type=subagent_type,
            description=description,
        )
        custom_event = CustomEvent(
            type=EventType.CUSTOM,
            name=TASK_SUBAGENT_EVENT_TYPE,
            value=payload.model_dump(),
        )
        return self._emit(custom_event, writer, namespace)



    # ---------- Human-in-the-loop interrupt ----------
    def hitl_interrupt(
        self,
        thread_id: str,
        interrupt: Any,
        metadata: Optional[Dict[str, Any]] = None,
        writer: Any = None,
        namespace: Optional[str] = None,
    ) -> CustomEvent:
        """Create a HITL interrupt custom event and optionally emit it."""
        payload = HITLInterruptEvent(
            thread_id=thread_id,
            interrupt=interrupt,
            metadata=metadata or {},
        )
        custom_event = CustomEvent(
            type=EventType.CUSTOM,
            name=HITL_INTERRUPT_EVENT_TYPE,
            value=payload.model_dump(),
        )
        return self._emit(custom_event, writer, namespace)


    # ---------- Sub-agent envelope ----------
    def subagent_event(
        self,
        *,
        task_id: str,
        subagent_namespace: Sequence[str],
        event: Dict[str, Any],
        writer: Any = None,
        namespace: Optional[str] = None,
    ) -> CustomEvent:
        """Wrap a normalized AG-UI event under a sub-agent task envelope."""
        payload = SubAgentEvent(
            task_id=task_id,
            namespace=list(subagent_namespace),
            event=event,
        )
        custom_event = CustomEvent(
            type=EventType.CUSTOM,
            name=SUBAGENT_EVENT_TYPE,
            value=payload.model_dump(),
        )
        return self._emit(custom_event, writer, namespace)


    # ---------- Before-agent event ----------
    def before_agent_event(
        self,
        *,
        message: str,
        metadata: Optional[Dict[str, Any]] = None,
        writer: Any = None,
        namespace: Optional[str] = None,
    ) -> CustomEvent:
        """Emit a before-agent custom event payload."""
        payload = BeforeAgentEvent(
            message=message,
            metadata=metadata or {},
        )
        custom_event = CustomEvent(
            type=EventType.CUSTOM,
            name=BEFORE_AGENT_EVENT_TYPE,
            value=payload.model_dump(),
        )
        return self._emit(custom_event, writer, namespace)



In [8]:
import json
from typing import Any, List, Optional, Tuple, Dict


_ALLOWED_MODES = {"messages", "updates"}

class AGUIStreamNormalizer:
    """
    Combined streaming normalizer for LangGraph:
        - stream_mode MUST be ["messages", "updates"] on the agent.
        - "messages" => ONLY assistant text content (chunks)
        - "updates"  => tools/subagents/plan/interrupt (NO content)

    Policies:
        - __interrupt__ => only HITL event (priority)
        - write_todos   => only plan_snapshot (dedupe)
        - task tool     => only subagent event (dedupe)
        - other tools   => full lifecycle start/args/result/end
    """

    def __init__(self, thread_id: str) -> None:
        self.emitter = AGUIEmitter()
        self.thread_id = thread_id  # message_id == thread_id (your requirement)

        # --- streaming state (per actor: orchestrator/sub-agent) ---
        self._stream_state: Dict[str, Dict[str, bool]] = {}

        # --- tool correlation ---
        self._pending_tool_call_ids: set[str] = set()     # start/args emitted; waiting ToolMessage result
        self._started_tool_call_ids: set[str] = set()     # to dedupe start/args
        self._finished_tool_call_ids: set[str] = set()    # to dedupe result/end
        self._ignored_tool_call_ids: set[str] = set()     # write_todos + task: ignore ToolMessage

        # --- custom event dedupe ---
        self._emitted_subagent_task_ids: set[str] = set()
        self._last_plan_fingerprint: Optional[str] = None

        # --- sub-agent namespace mapping ---
        self._pending_tasks: Dict[str, Dict[str, str]] = {}
        self._namespace_task_labels: Dict[tuple, str] = {}




    # --------------------------- public API ---------------------------
    def handle_chunk(self, chunk: Any) -> List[bytes]:
        """Convert a raw chunk to 0..N AG-UI SSE events (bytes)."""
        namespace, mode, payload, metadata = self._unwrap_envelope(chunk)

        # Handle payload based on mode
        if mode == "messages":
            events = self._handle_messages_payload(payload, metadata, namespace)
            return self._wrap_subagent_events_if_needed(events, namespace)
        elif mode == "updates":
            events = self._handle_updates_payload(payload, metadata, namespace)
            return self._wrap_subagent_events_if_needed(events, namespace)
        else:
            return []



    # ------------------------ envelope unwrapping ------------------------
    def _unwrap_envelope(
        self, chunk: Any
    ) -> Tuple[Optional[tuple], Optional[str], Any, Optional[Dict[str, Any]]]:
        """
        For stream_mode=["messages","updates"] with subgraphs=True possible.

        General rule:
        - Find the first string in the chunk that matches an allowed mode.
        - The item just before it (if any) is namespace.
        - The item just after it is payload.
        - The next item (if any) is metadata.
        Legacy: if messages payload is a 2-tuple (msg, meta) and meta is a dict,
        treat meta as metadata when none was provided.
        Fallback: dict chunks => updates; everything else => unknown.
        """
        namespace: Optional[tuple] = None
        mode: Optional[str] = None
        payload: Any = chunk
        metadata: Optional[Dict[str, Any]] = None

        # --- Sequence parsing: scan for mode string and derive neighbors ---
        if isinstance(chunk, (tuple, list)):
            seq = list(chunk)
            for idx, item in enumerate(seq):
                if isinstance(item, str) and item in _ALLOWED_MODES:
                    mode = item

                    # Namespace: immediately before the mode (if present)
                    if idx - 1 >= 0:
                        ns_candidate = seq[idx - 1]
                        if isinstance(ns_candidate, tuple) and len(ns_candidate) > 0:
                            namespace = ns_candidate
                        elif isinstance(ns_candidate, list) and len(ns_candidate) > 0:
                            namespace = tuple(ns_candidate)

                    # Payload: immediately after the mode (if present)
                    if idx + 1 < len(seq):
                        payload = seq[idx + 1]

                    # Metadata: the element after payload (if present)
                    if idx + 2 < len(seq):
                        meta_candidate = seq[idx + 2]
                        if isinstance(meta_candidate, dict):
                            metadata = meta_candidate

                    # Legacy: messages payload can be (msg, meta)
                    if (
                        mode == "messages"
                        and isinstance(payload, tuple)
                        and len(payload) == 2
                        and metadata is None
                    ):
                        msg, meta = payload
                        payload = msg
                        metadata = meta if isinstance(meta, dict) else None

                    return namespace, mode, payload, metadata

        # --- Fallbacks ---
        if isinstance(chunk, dict):
            return None, "updates", chunk, None

        return None, None, chunk, None



    # --------------------------- messages mode ---------------------------
    def _handle_messages_payload(
        self,
        payload: Any,
        metadata: Optional[Dict[str, Any]] = None,
        namespace: Optional[tuple] = None,
    ) -> List[bytes]:
        """
        messages:
            - AI message => assistant text stream (start/chunk)
            - ToolMessage => tool results (result/end) ONLY if tool_call_id is pending and not ignored
        """
        out: List[bytes] = []
        ns_label = self._resolve_namespace_label(namespace)

        kind = self._msg_kind(payload)

        # 1) ToolMessage in messages => treat as result/end (but gate via ignored/pending)
        if kind == "tool":
            self._emit_tool_message_result(out, payload, ns_label)
            return out

        # 2) AI message => assistant text chunks only
        if kind == "ai":
            delta = self._extract_text_delta(payload)
            if not delta:
                return out

            # First content => thinking_end + response_start + first chunk
            self._end_thinking_if_needed(out, ns_label)
            state = self._get_state(ns_label)

            if not state["response_started"]:
                self._push(out, self.emitter.response_start(message_id=self.thread_id, namespace=ns_label))
                state["response_started"] = True
                state["response_ended"] = False

            self._push(out, self.emitter.response_chunk(message_id=self.thread_id, delta=delta, namespace=ns_label))
            state["saw_messages_chunk"] = True
            return out

        # 3) Other message kinds ignored
        return out



    # --------------------------- updates mode ----------------------------
    def _handle_updates_payload(
        self,
        payload: Any,
        metadata: Optional[Dict[str, Any]] = None,
        namespace: Optional[tuple] = None,
    ) -> List[bytes]:
        """
        updates:
            - __interrupt__ => HITL event only (and return)
            - write_todos => plan_snapshot only (and mark tool_call_id ignored)
            - task => subagent event only (and mark tool_call_id ignored)
            - other tools => tool_start + tool_args (and mark tool_call_id pending)
            - If AI message contains content => DO NOT emit content; emit TEXT_MESSAGE_END (once)
        """
        out: List[bytes] = []
        if not isinstance(payload, dict):
            return out

        ns_label = self._resolve_namespace_label(namespace, payload)

        # HITL priority: HITL only contract
        if "__interrupt__" in payload:
            raw = payload.get("__interrupt__")
            interrupt_obj = raw[0] if isinstance(raw, (tuple, list)) and raw else raw

            interrupt_payload: Any = interrupt_obj
            if interrupt_obj is not None:
                interrupt_payload = {
                    "id": getattr(interrupt_obj, "id", None),
                    "value": getattr(interrupt_obj, "value", interrupt_obj),
                }

            hitl_meta: Dict[str, Any] = {}
            if isinstance(metadata, dict):
                hitl_meta.update(metadata)
            hitl_meta["namespace"] = ns_label

            self._push(
                out,
                self.emitter.hitl_interrupt(
                    thread_id=self.thread_id,
                    interrupt=interrupt_payload,
                    metadata=hitl_meta,
                    namespace=ns_label,
                ),
            )
            return out

        # Optional metadata forwarding (future-proof)
        meta: Dict[str, Any] = {}
        if isinstance(metadata, dict):
            meta.update(metadata)
        meta["namespace"] = ns_label

        # Walk node updates
        for node_name, node_update in payload.items():
            if node_update is None or not isinstance(node_update, dict):
                continue

            # Emit a dedicated marker for sub-agent before_agent instructions.
            if (
                namespace is not None
                and node_name == "PatchToolCallsMiddleware.before_agent"
            ):
                for msg in self._unwrap_messages_list(node_update.get("messages")):
                    delegated_message = getattr(msg, "content", None)
                    if isinstance(delegated_message, str) and delegated_message:
                        self._push(
                            out,
                            self.emitter.before_agent_event(
                                message=delegated_message,
                                metadata=meta,
                                namespace=ns_label,
                            ),
                        )
                        break

            # Authoritative todos snapshot may be present as node_update["todos"]
            if "todos" in node_update and isinstance(node_update.get("todos"), list):
                fp = self._fingerprint(node_update["todos"])
                if fp != self._last_plan_fingerprint:
                    self._end_thinking_if_needed(out, ns_label)
                    self._push(out, self.emitter.plan_snapshot(node_update["todos"], metadata=meta, namespace=ns_label))
                    self._last_plan_fingerprint = fp

            # Process messages inside this update
            for msg in self._unwrap_messages_list(node_update.get("messages")):
                msg_kind = self._msg_kind(msg)

                # ToolMessage results may also arrive through updates-only streams.
                if msg_kind == "tool":
                    self._emit_tool_message_result(out, msg, ns_label)
                    continue

                if msg_kind != "ai":
                    continue

                # If updates stream contains final AI content:
                # - With messages-mode chunks already seen: only close response.
                # - With updates-only mode: synthesize start/content/end once.
                ai_content = self._extract_text_delta(msg)
                if ai_content:
                    self._end_thinking_if_needed(out, ns_label)
                    state = self._get_state(ns_label)

                    if not state["response_started"]:
                        self._push(out, self.emitter.response_start(message_id=self.thread_id, namespace=ns_label))
                        state["response_started"] = True
                        state["response_ended"] = False

                    if not state["saw_messages_chunk"]:
                        self._push(out, self.emitter.response_content(message_id=self.thread_id, delta=ai_content, namespace=ns_label))

                    if not state["response_ended"]:
                        self._push(out, self.emitter.response_end(message_id=self.thread_id, namespace=ns_label))
                        state["response_ended"] = True

                # Tool intents: emit start/args OR plan/subagent events
                for tc in self._iter_tool_calls(msg):
                    tc_id = tc["id"]
                    tc_name = tc["name"]
                    tc_args = tc.get("args")

                    if tc_name == "write_todos":
                        # plan snapshot only
                        todos = tc_args.get("todos") if isinstance(tc_args, dict) else None
                        if isinstance(todos, list):
                            fp = self._fingerprint(todos)
                            if fp != self._last_plan_fingerprint:
                                self._end_thinking_if_needed(out, ns_label)
                                self._push(out, self.emitter.plan_snapshot(todos, metadata=meta, namespace=ns_label))
                                self._last_plan_fingerprint = fp
                        self._ignored_tool_call_ids.add(tc_id)  # ignore ToolMessage if it appears later
                        continue

                    if tc_name == "task":
                        # subagent event only
                        if tc_id not in self._emitted_subagent_task_ids and isinstance(tc_args, dict):
                            subagent_type = str(tc_args.get("subagent_type", ""))
                            description = str(tc_args.get("description", ""))
                            self._end_thinking_if_needed(out, ns_label)
                            self._push(
                                out,
                                self.emitter.task_subagent(
                                    task_id=tc_id,
                                    subagent_type=subagent_type,
                                    description=description,
                                    namespace=ns_label,
                                ),
                            )
                            self._emitted_subagent_task_ids.add(tc_id)
                            self._pending_tasks[tc_id] = {
                                "description": description,
                                "subagent_type": subagent_type,
                            }
                        self._ignored_tool_call_ids.add(tc_id)  # ignore ToolMessage if it appears later
                        continue

                    # Normal tool => start + args, and wait for ToolMessage on messages stream
                    if tc_id in self._started_tool_call_ids:
                        continue

                    self._end_thinking_if_needed(out, ns_label)
                    self._push(out, self.emitter.tool_call_start(tool_call_id=tc_id, name=tc_name, namespace=ns_label))
                    self._push(out, self.emitter.tool_call_args(tool_call_id=tc_id, name=tc_name, args=tc_args, namespace=ns_label))

                    self._started_tool_call_ids.add(tc_id)
                    self._pending_tool_call_ids.add(tc_id)

        return out


    def _emit_tool_message_result(self, out: List[bytes], msg: Any, ns_label: Optional[str]) -> None:
        """Emit tool result/end for tool messages we previously opened."""
        tool_call_id = getattr(msg, "tool_call_id", None)
        tool_output = getattr(msg, "content", "")

        if not tool_call_id:
            return

        # Ignore results for write_todos/task tool calls (because we never emitted start/args)
        if tool_call_id in self._ignored_tool_call_ids:
            self._ignored_tool_call_ids.discard(tool_call_id)
            return

        # Only close tools we actually started from updates
        if tool_call_id in self._pending_tool_call_ids and tool_call_id not in self._finished_tool_call_ids:
            self._end_thinking_if_needed(out, ns_label)
            self._push(
                out,
                self.emitter.tool_call_result(
                    tool_call_id=tool_call_id,
                    thread_id=self.thread_id,
                    output=tool_output,
                    namespace=ns_label,
                ),
            )
            self._push(
                out,
                self.emitter.tool_call_end(tool_call_id=tool_call_id, namespace=ns_label),
            )

            self._pending_tool_call_ids.discard(tool_call_id)
            self._finished_tool_call_ids.add(tool_call_id)



    # ------------------------- utilities & helpers -------------------------
    def _push(self, events: List[bytes], maybe_event: Optional[bytes]) -> None:
        # Emitter methods return Optional[bytes]
        if maybe_event is not None:
            events.append(maybe_event)


    def _extract_text_delta(self, msg: Any) -> str:
        """
        Extract assistant text delta from a LangChain message chunk.
        Handles common shapes:
            - msg.content: str
            - msg.content: list[dict] with 'text' fields (e.g., rich content parts)
        """
        content = getattr(msg, "content", "")
        if isinstance(content, str):
            return content

        # Sometimes content can be a list of parts; we only keep textual pieces
        if isinstance(content, list):
            parts: List[str] = []
            for p in content:
                if isinstance(p, dict):
                    # common keys: {"type":"text","text":"..."} or {"text":"..."}
                    text = p.get("text")
                    if isinstance(text, str) and text:
                        parts.append(text)
            return "".join(parts)

        return ""


    def _end_thinking_if_needed(self, out: List[bytes], namespace: Optional[str]) -> None:
        # We do NOT call this for HITL (because you want HITL only).
        state = self._get_state(namespace)
        if state["thinking_started"]:
            self._push(out, self.emitter.thinking_end(namespace=namespace))
            state["thinking_started"] = False


    def _actor_key(self, namespace: Optional[str]) -> str:
        """Build a stable stream-state key per orchestrator/sub-agent."""
        return namespace if isinstance(namespace, str) and namespace else "__orchestrator__"


    def _get_state(self, namespace: Optional[str]) -> Dict[str, bool]:
        """Return mutable stream state for the given actor key."""
        key = self._actor_key(namespace)
        if key not in self._stream_state:
            self._stream_state[key] = {
                "response_started": False,
                "response_ended": False,
                "thinking_started": False,
                "saw_messages_chunk": False,
            }
        return self._stream_state[key]


    def _unwrap_messages_list(self, messages_obj: Any) -> List[Any]:
        """
        updates often store messages as:
            - list[Message]
            - Overwrite(value=[Message, ...])
            - single Message
        """
        if messages_obj is None:
            return []

        # Overwrite(value=[...]) case
        value = getattr(messages_obj, "value", None)
        if isinstance(value, list):
            return value

        if isinstance(messages_obj, list):
            return messages_obj

        # Single message object fallback
        return [messages_obj]


    def _iter_tool_calls(self, ai_msg: Any) -> List[Dict[str, Any]]:
        """
        Returns tool_calls as a list of dicts: {id,name,args}
        """
        tool_calls = getattr(ai_msg, "tool_calls", None)

        # Some wrappers put tool calls in additional_kwargs
        if not tool_calls:
            ak = getattr(ai_msg, "additional_kwargs", None)
            if isinstance(ak, dict):
                tool_calls = ak.get("tool_calls")

        if not tool_calls:
            return []

        normalized: List[Dict[str, Any]] = []
        for tc in tool_calls:
            if isinstance(tc, dict):
                normalized.append(
                    {"id": tc.get("id"), "name": tc.get("name"), "args": tc.get("args")}
                )
            else:
                normalized.append(
                    {
                        "id": getattr(tc, "id", None),
                        "name": getattr(tc, "name", None),
                        "args": getattr(tc, "args", None),
                    }
                )
        return [t for t in normalized if t.get("id") and t.get("name")]


    def _maybe_bind_namespace(self, namespace: Optional[tuple], payload: Any) -> Optional[str]:
        """
        Map a LangGraph subgraph namespace (tuple) to a task_id by matching the
        first HumanMessage content of PatchToolCallsMiddleware.before_agent to a
        pending task description.
        """
        if namespace is None or not isinstance(payload, dict):
            return None

        if namespace in self._namespace_task_labels:
            return self._namespace_task_labels[namespace]

        node = payload.get("PatchToolCallsMiddleware.before_agent")
        if not isinstance(node, dict):
            return None

        for msg in self._unwrap_messages_list(node.get("messages")):
            content = getattr(msg, "content", None)
            if not isinstance(content, str) or not content:
                continue

            for task_id, info in list(self._pending_tasks.items()):
                if content == info.get("description"):
                    task_str = str(task_id)
                    self._namespace_task_labels[namespace] = task_str
                    self._pending_tasks.pop(task_id, None)
                    return task_str

        return None


    def _resolve_namespace_label(self, namespace: Optional[tuple], payload: Any = None) -> Optional[str]:
        """
        Return the mapped task_id for a namespace (if any). When payload is provided,
        attempt to bind first (for sub-agent startup chunks).
        """
        if namespace is None:
            return None

        if payload is not None:
            self._maybe_bind_namespace(namespace, payload)

        # Prefer explicit mapping (task tool-call id), then deterministic namespace-derived id.
        return self._namespace_task_labels.get(namespace) or self._namespace_task_id(namespace)


    def _wrap_subagent_events_if_needed(self, events: List[bytes], namespace: Optional[tuple]) -> List[bytes]:
        """
        For sub-agent namespaces, wrap every normalized AG-UI event inside SUBAGENT_EVENT.
        Orchestrator events (namespace None/empty) pass through unchanged.
        """
        task_id = self._namespace_task_id(namespace)
        if task_id is None:
            return events

        namespace_path = self._namespace_path(namespace)
        namespace_token = self._namespace_token(namespace)
        wrapped: List[bytes] = []

        for event_bytes in events:
            inner_event = self._sse_to_payload(event_bytes)
            if inner_event is None:
                inner_event = self._raw_event_payload(event_bytes)

            wrapped_event = self.emitter.subagent_event(
                task_id=task_id,
                subagent_namespace=namespace_path or [task_id],
                event=inner_event,
                namespace=namespace_token,
            )
            if wrapped_event is not None:
                wrapped.append(wrapped_event)

        return wrapped


    def _namespace_task_id(self, namespace: Optional[tuple]) -> Optional[str]:
        """
        Deterministically derive task_id from namespace.
        Example: ('tools:abc-123',) -> 'abc-123'
        """
        if namespace is None:
            return None

        if isinstance(namespace, (list, tuple)) and len(namespace) == 0:
            return None

        parts = namespace if isinstance(namespace, (list, tuple)) else [namespace]
        for part in parts:
            if not isinstance(part, str) or not part:
                continue
            if ":" in part:
                _, _, tail = part.partition(":")
                return tail or part
            return part
        return None


    def _namespace_path(self, namespace: Optional[tuple]) -> Optional[List[str]]:
        """Return a serialized namespace path for diagnostics/UI correlation."""
        if namespace is None:
            return None

        if isinstance(namespace, (list, tuple)):
            path = [str(item) for item in namespace if str(item)]
            return path or None

        text = str(namespace)
        return [text] if text else None


    def _namespace_token(self, namespace: Optional[tuple]) -> Optional[str]:
        """
        Return the first namespace token for transport-level namespace field.
        Example: ('tools:abc', 'model:def') -> 'tools:abc'
        """
        path = self._namespace_path(namespace)
        if not path:
            return None
        return path[0]


    def _sse_to_payload(self, sse_event: bytes) -> Optional[Dict[str, Any]]:
        """Decode the JSON payload from an SSE frame."""
        try:
            text = sse_event.decode("utf-8")
        except Exception:
            return None

        for line in text.splitlines():
            if line.startswith("data:"):
                raw = line[len("data:"):].lstrip()
                try:
                    payload = json.loads(raw)
                except Exception:
                    return None
                if isinstance(payload, dict):
                    return payload
                return None

        return None


    def _raw_event_payload(self, sse_event: bytes) -> Dict[str, Any]:
        """
        Fallback payload when an SSE frame cannot be decoded into JSON.
        Ensures sub-agent events are still wrapped deterministically.
        """
        try:
            text = sse_event.decode("utf-8")
        except Exception:
            text = repr(sse_event)

        return {
            "type": "RAW_SSE_EVENT",
            "raw_sse": text,
        }


    def _fingerprint(self, obj: Any) -> str:
        # Stable JSON fingerprint for dedupe (todos lists, etc.)
        return json.dumps(obj, ensure_ascii=False, sort_keys=True)


    def _msg_kind(self, msg: Any) -> str:
        """
        Duck-typing classifier for LangChain messages/chunks.
        """
        cls = msg.__class__.__name__
        role = getattr(msg, "role", None)
        mtype = getattr(msg, "type", None)

        # ToolMessage is reliably detectable via tool_call_id
        if getattr(msg, "tool_call_id", None) is not None or role == "tool" or mtype == "tool" or cls == "ToolMessage":
            return "tool"

        if role == "assistant" or mtype in ("ai", "assistant") or "AIMessage" in cls:
            return "ai"

        return "other"



## Deep Agent - Simple

In [6]:
model = init_chat_model("gpt-5")

In [7]:
@tool
def toy_web_search(query: str) -> str:
    """Toy web search tool (stub)."""
    return (
        "RESULT 1: (stub) LangGraph is a graph-based orchestration framework.\n"
        "RESULT 2: (stub) Deep agents use planning + tools + subagents.\n"
        f"QUERY WAS: {query}"
    )

@tool
def toy_arxiv_search(query: str) -> str:
    """Toy arXiv search tool (stub)."""
    return (
        "PAPER A: (stub) Tool-use agents survey\n"
        "PAPER B: (stub) Planning and reflection in agents\n"
        f"QUERY WAS: {query}"
    )

In [ ]:
system_prompt = """
You are a helpful research assistant.

For non-trivial requests:
1) Use write_todos to create a short plan.
2) Use tools when useful.
3) Produce a concise final answer.
"""

agent = create_deep_agent(
    model=model,
    tools=[toy_web_search, toy_arxiv_search],
    system_prompt=system_prompt,
)

In [19]:
await run_stream(
    "Research LangGraph briefly. Make a short plan first, then use both tools, then answer.",
    stream_mode="messages",
    deep_agent=agent,
)

(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--e6bb9fcd-4bfd-43f7-8052-ba1dcaf3a497', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_4JsPM8hpUzoYNxkhgtNkoQ3C', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_4JsPM8hpUzoYNxkhgtNkoQ3C', 'index': 0, 'type': 'tool_call_chunk'}]), {'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:4d61cd56-6f9f-b1ec-9f08-f23d8af1316c', 'checkpoint_ns': 'model:4d61cd56-6f9f-b1ec-9f08-f23d8af1316c', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--e6bb9fcd-4bfd-43f7-8052-ba1dcaf3a497', tool_calls=[{'name': '', 'args': {}, 'id': None, 'type': 'tool_call'}], too

In [22]:
await run_stream(
    "Research LangGraph briefly. Make a short plan first, then use both tools, then answer.",
    stream_mode="updates",
    deep_agent=agent,
)

{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='b90d847e-a016-4a9d-8a3d-bfa7219a42f5')])}}
{'SummarizationMiddleware.before_model': None}
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 4508, 'total_tokens': 4555, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 4352}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c4585b5b9c', 'id': 'chatcmpl-CvtA2FGNsVQ80eTOnGsqEsY8DWCBu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--a60cff1a-3b95-4c98-9ade-09413450b8a2-0

In [23]:
await run_stream(
    "Research LangGraph briefly. Make a short plan first, then use both tools, then answer.",
    stream_mode=["messages", "updates"],
    deep_agent=agent,
)

('updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='e9c1842b-d939-4fe2-972f-fc3f0ac608f4')])}})
('updates', {'SummarizationMiddleware.before_model': None})
('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--b5b9aa1f-e5d4-4aad-a116-3a80c696b7b8', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_FsiqmKKNtyybx7YVabhcXkxC', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_FsiqmKKNtyybx7YVabhcXkxC', 'index': 0, 'type': 'tool_call_chunk'}]), {'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:508c95c3-afc4-0d42-7c81-58cce9b6e800', 'checkpoint_ns': 'model:508c95c3-a

## Deep Agent - Sub Agents

In [25]:
model = init_chat_model("openai:gpt-4o-mini")

In [26]:
@tool
def get_weather_dummy(city: str) -> str:
    """Return a dummy weather report for a city."""
    return f"[DUMMY WEATHER] {city}: Sunny, 25°C."

@tool
def get_crypto_dummy(symbol: str) -> str:
    """Return a dummy crypto price for a symbol."""
    symbol = symbol.upper().strip()
    return f"[DUMMY CRYPTO] {symbol}: $42,000.00."

@tool
def it_services_dummy(request: str) -> str:
    """Return a dummy IT services response."""
    return "[DUMMY IT] We offer 24/7 monitoring, backups, and incident response. ETA: same-day onboarding."

@tool
def marketing_dummy(product: str) -> str:
    """Return a dummy marketing tagline."""
    return f"[DUMMY MARKETING] '{product}': Secure. Fast. Done."

In [27]:
subagents = [
    {
        "name": "weather",
        "description": "Answers weather questions.",
        "system_prompt": (
            "You are the Weather subagent. Always call get_weather_dummy once, "
            "then return a single concise sentence to the user."
        ),
        "tools": [get_weather_dummy],
    },
    {
        "name": "crypto",
        "description": "Answers cryptocurrency questions.",
        "system_prompt": (
            "You are the Crypto subagent. Always call get_crypto_dummy once, "
            "then return a single concise sentence to the user."
        ),
        "tools": [get_crypto_dummy],
    },
    {
        "name": "it-services",
        "description": "Answers IT services / support questions.",
        "system_prompt": (
            "You are the IT Services subagent. Always call it_services_dummy once, "
            "then return a short, practical answer."
        ),
        "tools": [it_services_dummy],
    },
    {
        "name": "marketer",
        "description": "Creates short marketing copy.",
        "system_prompt": (
            "You are the Marketer subagent. Always call marketing_dummy once, "
            "then return 1 tagline + 2 bullet benefits."
        ),
        "tools": [marketing_dummy],
    },
]

In [28]:
SYSTEM_PROMPT = """
You are a supervisor deep agent.
When the user asks about:
- weather -> delegate to subagent 'weather'
- crypto prices -> delegate to subagent 'crypto'
- IT services -> delegate to subagent 'it-services'
- marketing / copy -> delegate to subagent 'marketer'

Use the built-in task tool to delegate.
If multiple domains are requested, delegate to each relevant subagent and then merge results.
Keep the final answer short.
"""

In [29]:
agent = create_deep_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    subagents=subagents,
)

In [30]:
await run_stream(
    "Weather in Athens, BTC price, and give me a tagline for 'Kostas IT Care'. Also what IT services do you offer?",
    stream_mode="updates",
    deep_agent=agent,
)

{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content="Weather in Athens, BTC price, and give me a tagline for 'Kostas IT Care'. Also what IT services do you offer?", additional_kwargs={}, response_metadata={}, id='a7277898-927b-4a3e-9629-04a9a845177b')])}}
{'SummarizationMiddleware.before_model': None}
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 4516, 'total_tokens': 4643, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bbc38b4db', 'id': 'chatcmpl-Cw64p2yqDKzWfTRzUTTa9UIxDBMKE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--520f54d7-c581-49c

In [31]:
await run_stream(
    "Weather in Athens, BTC price, and give me a tagline for 'Kostas IT Care'. Also what IT services do you offer?",
    stream_mode="messages",
    deep_agent=agent,
)

(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--1bd815b6-3d42-4910-b277-0d001f1598e6'), {'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:1ad3c3f0-481d-c6be-31f1-9742ef40f310', 'checkpoint_ns': 'model:1ad3c3f0-481d-c6be-31f1-9742ef40f310', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--1bd815b6-3d42-4910-b277-0d001f1598e6', tool_calls=[{'name': 'task', 'args': {}, 'id': 'call_YjgGCCSgVsFW1KxznINfxkZH', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'task', 'args': '', 'id': 'call_YjgGCCSgVsFW1KxznINfxkZH', 'index': 0, 'type': 'tool_call_chunk'}]), {'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model

In [32]:
await run_stream(
    "Weather in Athens, BTC price, and give me a tagline for 'Kostas IT Care'. Also what IT services do you offer?",
    stream_mode=["messages", "updates"],
    deep_agent=agent,
)

('updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content="Weather in Athens, BTC price, and give me a tagline for 'Kostas IT Care'. Also what IT services do you offer?", additional_kwargs={}, response_metadata={}, id='ce195282-ea15-4c6e-bad9-440c15e18c6b')])}})
('updates', {'SummarizationMiddleware.before_model': None})
('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--32d6b67b-749a-4f9e-87b6-847367b6df99'), {'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:2d640c5f-b0f6-fe2a-2737-a9acdcf2dc99', 'checkpoint_ns': 'model:2d640c5f-b0f6-fe2a-2737-a9acdcf2dc99', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None}))
('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'m

## Deep Agent - HITL & Sub Agents

In [56]:
@tool
def toy_web_search(query: str) -> str:
    """Toy web search tool (stub)."""
    return (
        "RESULT 1: (stub) LangGraph is a graph-based orchestration framework.\n"
        "RESULT 2: (stub) Deep agents use planning + tools + subagents.\n"
        f"QUERY WAS: {query}"
    )

@tool
def toy_arxiv_search(query: str) -> str:
    """Toy arXiv search tool (stub)."""
    return (
        "PAPER A: (stub) Tool-use agents survey\n"
        "PAPER B: (stub) Planning and reflection in agents\n"
        f"QUERY WAS: {query}"
    )

In [ ]:
researchSubagent = {
    "name": "research-agent",
    "description": "Does focused research using toy_web_search + toy_arxiv_search and returns a short synthesis.",
    "system_prompt": """
        You are a research subagent.
            - Use BOTH tools at least once when asked to research.
            - Keep intermediate junk out of your final response.
        Output:
            1) 3-5 bullet findings
            2) 1 short paragraph synthesis
    """,
    # Keep toolset minimal for focus + safety
    "tools": [toy_web_search, toy_arxiv_search],
    # Subagent-level HITL override example (no edit allowed here)
    "interrupt_on": {
        "toy_arxiv_search": {"allowed_decisions": ["approve", "reject"]},
    },
}

In [58]:
model = init_chat_model("openai:gpt-4o-mini")

systemPrompt = """You are a helpful research assistant.

Rules:
1) For non-trivial requests: use write_todos to create a short plan.
2) Delegate research-heavy work to the research-agent using:
    task(name="research-agent", task="...")
3) Use tools when useful and keep the final answer concise.
"""

In [59]:
checkpointer = MemorySaver()

In [101]:
agent = create_deep_agent(
    model=model,
    # tools=[toy_web_search, toy_arxiv_search],
    system_prompt=systemPrompt,
    subagents=[researchSubagent],
    interrupt_on={
        # HITL on the main agent: allow approve/edit/reject (default behavior when True) :contentReference[oaicite:5]{index=5}
        "toy_web_search": True,
    },
    checkpointer=checkpointer,
)

In [55]:
await run_stream(
    "Research LangGraph briefly. Make a short plan, delegate research to research-agent, then answer.",
    stream_mode="updates",
    deep_agent=agent,
)

{'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan, delegate research to research-agent, then answer.', additional_kwargs={}, response_metadata={}, id='507b1780-b46b-4e7e-a0b9-4467ae3070cd')])}}
{'SummarizationMiddleware.before_model': None}
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2572, 'prompt_tokens': 4598, 'total_tokens': 7170, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2496, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Cw7kNRyi4W0X6czz4u7Rdkyvvv2lJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--f72cbbc5-d4c7-48bb-ace2-539066a86962-0', to

In [62]:
await run_stream(
    "Research LangGraph briefly. Make a short plan, delegate research to research-agent, then answer.",
    stream_mode="messages",
    deep_agent=agent,
)

(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--b6f13ae3-cd47-4dbd-8e7a-2ad22951fab3', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_UN2JuVvjSJrTe7jzldta7orP', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_UN2JuVvjSJrTe7jzldta7orP', 'index': 0, 'type': 'tool_call_chunk'}]), {'thread_id': 'c9e303f2-4edb-4e30-ab91-7c15b2314dfb', 'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:d3efc03c-efdb-0b88-5a12-1d2dfdd55036', 'checkpoint_ns': 'model:d3efc03c-efdb-0b88-5a12-1d2dfdd55036', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--b6f13ae3-cd47-4dbd-8e7a-2ad22951fab3', tool_calls=[{'name': '

In [63]:
await run_stream(
    "Research LangGraph briefly. Make a short plan, delegate research to research-agent, then answer.",
    stream_mode=["messages", "updates"],
    deep_agent=agent,
)

('updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan, delegate research to research-agent, then answer.', additional_kwargs={}, response_metadata={}, id='e81e0b25-e410-4a94-be95-732a69225ec3')])}})
('updates', {'SummarizationMiddleware.before_model': None})
('messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--04a141cc-6a24-406c-bc38-9a6e9bc64183', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_S2whwccOq1kHXwd0O1rNSmAB', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_S2whwccOq1kHXwd0O1rNSmAB', 'index': 0, 'type': 'tool_call_chunk'}]), {'thread_id': '8b70f0b7-a99f-4124-941a-a700f60c48e0', 'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:d8653975-2

In [83]:
await run_stream_unwrap(
    "Research LangGraph briefly. Make a short plan first, then use both tools, then answer.",
    stream_mode=["updates"],
    deep_agent=agent,
)

Namespace: None
Mode: updates
Payload: {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='f56fb99a-4249-4926-8fc5-d820caecebb3')])}}
Metadata: None
Namespace: None
Mode: updates
Payload: {'SummarizationMiddleware.before_model': None}
Metadata: None
Namespace: None
Mode: updates
Payload: {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 4514, 'total_tokens': 4564, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 3968}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bbc38b4db', 'id': 'chatcmpl-CwHCGE6SoKhs

In [84]:
await run_stream_unwrap(
    "Research LangGraph briefly. Make a short plan first, then use both tools, then answer.",
    stream_mode="messages",
    deep_agent=agent,
)

Namespace: None
Mode: messages
Payload: content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--d51bd7df-408d-401f-9989-75c7a4ac9f03' tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_SDqKknEgkg0XeJuV4Ca3LFPC', 'type': 'tool_call'}] tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_SDqKknEgkg0XeJuV4Ca3LFPC', 'index': 0, 'type': 'tool_call_chunk'}]
Metadata: {'thread_id': 'fe6ce014-d57c-4703-b0bd-ebee9dce8764', 'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:5803009a-8626-0c04-a9f5-913a1a98f995', 'checkpoint_ns': 'model:5803009a-8626-0c04-a9f5-913a1a98f995', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None}
Namespace: None
Mode: messages
Payload: content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--d51bd7df-408d-

## Deep Agent - AGUI Handling & Long Term Memory

In [9]:

@tool
def toy_web_search(query: str) -> str:
    """Toy web search tool (stub)."""
    return (
        "RESULT 1: (stub) LangGraph is a graph-based orchestration framework.\n"
        "RESULT 2: (stub) Deep agents use planning + tools + subagents.\n"
        f"QUERY WAS: {query}"
    )

@tool
def toy_arxiv_search(query: str) -> str:
    """Toy arXiv search tool (stub)."""
    return (
        "PAPER A: (stub) Tool-use agents survey\n"
        "PAPER B: (stub) Planning and reflection in agents\n"
        f"QUERY WAS: {query}"
    )


researchSubagent = {
    "namespace": "research-agent",
    "name": "research-agent",
    "description": "Does focused research using toy_web_search + toy_arxiv_search and returns a short synthesis.",
    "system_prompt": """
        You are a research subagent.
            - Use BOTH tools at least once when asked to research.
            - Keep intermediate junk out of your final response.
        Output:
            1) 3-5 bullet findings
            2) 1 short paragraph synthesis
    """,
    # Keep toolset minimal for focus + safety
    "tools": [toy_web_search, toy_arxiv_search],
}

model = init_chat_model("openai:gpt-4o-mini")

systemPrompt = """You are a helpful research assistant.

Rules:
1) For non-trivial requests: use write_todos to create a short plan.
2) Delegate research-heavy work to the research-agent using:
    task(name="research-agent", task="...")
3) Use tools when useful and keep the final answer concise.
"""

checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    system_prompt=systemPrompt,
    subagents=[researchSubagent],
    checkpointer=checkpointer,
)

In [10]:
configurable = {"configurable": {"thread_id": str(uuid.uuid4())}}
agui_normalizer = AGUIStreamNormalizer(thread_id=configurable["configurable"]["thread_id"])
inputs = {"messages": [{"role": "user", "content": "Research LangGraph briefly. Make a short plan first, then use both tools, then answer."}]}

async for chunk in agent.astream(
    inputs,
    stream_mode=["updates", "messages"],
    config=configurable,
    subgraphs=True,
):
    print(chunk)
    for sse_event in agui_normalizer.handle_chunk(chunk):
        print(sse_event)
    
    print("-----\n\n")

((), 'updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='5e24aa0a-de93-46bc-8032-0ba2101efd28')])}})
-----


((), 'updates', {'SummarizationMiddleware.before_model': None})
-----


((), 'messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--8460f177-b453-41d2-a9b8-f6f82e3cd9b8', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_f1F7VGmqXWDoUAseOStCzGVb', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_f1F7VGmqXWDoUAseOStCzGVb', 'index': 0, 'type': 'tool_call_chunk'}]), {'thread_id': 'b8cd4edd-72b9-4db5-a060-f4e0bc88c84d', 'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns':

In [11]:
configurable = {"configurable": {"thread_id": str(uuid.uuid4())}}
agui_normalizer = AGUIStreamNormalizer(thread_id=configurable["configurable"]["thread_id"])
inputs = {"messages": [{"role": "user", "content": "Research LangGraph briefly. Make a short plan first, then use both tools, then answer."}]}

async for chunk in agent.astream(
    inputs,
    stream_mode=["updates"],
    config=configurable,
    subgraphs=True,
):
    print(chunk)
    namespace, mode, payload, metadata = agui_normalizer._unwrap_envelope(chunk)
    print("Namespace:", namespace)
    print("Mode:", mode)
    print("Payload:", payload)
    print("Metadata:", metadata)
    for sse_event in agui_normalizer.handle_chunk(chunk):
        print(sse_event)
    print("-----\n\n")

((), 'updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='a7d31338-8788-4cb9-aa01-f47597959a21')])}})
Namespace: None
Mode: updates
Payload: {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='a7d31338-8788-4cb9-aa01-f47597959a21')])}}
Metadata: None
-----


((), 'updates', {'SummarizationMiddleware.before_model': None})
Namespace: None
Mode: updates
Payload: {'SummarizationMiddleware.before_model': None}
Metadata: None
-----


((), 'updates', {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 4467, 'total_tokens': 45

In [12]:
configurable = {"configurable": {"thread_id": str(uuid.uuid4())}}
agui_normalizer = AGUIStreamNormalizer(thread_id=configurable["configurable"]["thread_id"])
inputs = {"messages": [{"role": "user", "content": "Research LangGraph briefly. Make a short plan first, then use both tools, then answer."}]}

async for chunk in agent.astream(
    inputs,
    stream_mode=["messages"],
    config=configurable,
    subgraphs=True,
):
    print(chunk)
    for sse_event in agui_normalizer.handle_chunk(chunk):
        print(sse_event)

((), 'messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--b01b6e7e-94b0-4c4c-801b-1c859f80906d', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_Mb4tXC82tcRkUtTepsIRmRze', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_Mb4tXC82tcRkUtTepsIRmRze', 'index': 0, 'type': 'tool_call_chunk'}]), {'thread_id': '53f3a2a9-5cf5-4bba-a8af-78e42cd311e6', 'langgraph_step': 3, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:cca886cb-ed9e-0827-e045-3d8dcb91b602', 'checkpoint_ns': 'model:cca886cb-ed9e-0827-e045-3d8dcb91b602', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None}))
((), 'messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--b01b6e7e-94b0-4c4c-801b-1c

In [ ]:
from pathlib import Path
from pprint import pformat
from typing import Optional
import ast
import json


AGUI_EVENTS_DIR = Path("data/deep_agents_agui")
AGUI_EVENTS_DIR.mkdir(parents=True, exist_ok=True)


def _agui_event_to_text(sse_event: Any) -> str:
    if isinstance(sse_event, bytes):
        try:
            return sse_event.decode("utf-8")
        except Exception:
            return repr(sse_event)
    return str(sse_event)


def _agui_event_to_payload(sse_event: Any) -> Dict[str, Any]:
    event_text = _agui_event_to_text(sse_event)

    for line in event_text.splitlines():
        if line.startswith("data:"):
            raw_payload = line[len("data:"):].lstrip()
            payload = json.loads(raw_payload)
            if isinstance(payload, dict):
                return payload
            break

    raise ValueError(f"Could not decode AGUI event payload: {event_text}")


async def run_agui_handling_with_memory(user_text: str) -> tuple[str, Path, list[Dict[str, Any]]]:
    thread_id = str(uuid.uuid4())
    configurable = {"configurable": {"thread_id": thread_id}}
    agui_normalizer = AGUIStreamNormalizer(thread_id=thread_id)
    inputs = {"messages": [{"role": "user", "content": user_text}]}
    agui_events: list[Dict[str, Any]] = []

    async for chunk in agent.astream(
        inputs,
        stream_mode=["messages", "updates"],
        config=configurable,
        subgraphs=True,
    ):
        print(chunk)
        for sse_event in agui_normalizer.handle_chunk(chunk):
            event_text = _agui_event_to_text(sse_event)
            print(event_text)
            agui_events.append(_agui_event_to_payload(sse_event))

        print("-----\\n\\n")

    output_path = AGUI_EVENTS_DIR / f"{thread_id}.py"
    output_path.write_text(
        "agui_events = " + pformat(agui_events, width=120, sort_dicts=False),
        encoding="utf-8",
    )

    print(f"Saved {len(agui_events)} AGUI events to {output_path}")
    return thread_id, output_path, agui_events


def read_saved_agui_events(*, thread_id: Optional[str] = None, path: Optional[Path] = None) -> list[Dict[str, Any]]:
    if path is None:
        if not thread_id:
            raise ValueError("Provide either 'thread_id' or 'path'.")
        path = AGUI_EVENTS_DIR / f"{thread_id}.py"

    source = path.read_text(encoding="utf-8")
    prefix = "agui_events ="
    if not source.startswith(prefix):
        raise ValueError(f"Expected file to start with '{prefix}' in {path}")

    literal = source[len(prefix):].strip()
    if literal.endswith("\\n"):
        literal = literal[:-2].rstrip()

    agui_events = ast.literal_eval(literal)
    if not isinstance(agui_events, list):
        raise TypeError(f"Expected 'agui_events' to be a list in {path}")

    return agui_events


def preview_saved_agui_events(*, thread_id: Optional[str] = None, path: Optional[Path] = None, limit: int = 3) -> list[Dict[str, Any]]:
    agui_events = read_saved_agui_events(thread_id=thread_id, path=path)
    print(f"Loaded {len(agui_events)} AGUI events")
    for index, event in enumerate(agui_events[:limit], start=1):
        print(f"[{index}] {json.dumps(event, ensure_ascii=False, indent=2)}")
    return agui_events


In [17]:
thread_id_1, output_path_1, agui_events_1 = await run_agui_handling_with_memory(
    "Research LangGraph briefly. Make a short plan first, then use both tools, then answer."
)

print("thread_id_1:", thread_id_1)
print("output_path_1:", output_path_1)
print("agui_events_1 count:", len(agui_events_1))

((), 'updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Research LangGraph briefly. Make a short plan first, then use both tools, then answer.', additional_kwargs={}, response_metadata={}, id='b954d313-bc7d-4e7e-a52c-c0a303dc591d')])}})
-----\n\n
((), 'messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019d1010-1cc7-74f0-b7f2-beedb652e319', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_Pjs0iPx4eYZT10dVqtDNdI58', 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_Pjs0iPx4eYZT10dVqtDNdI58', 'index': 0, 'type': 'tool_call_chunk'}]), {'ls_integration': 'langchain_chat_model', 'thread_id': 'b559e8e4-2d06-4719-b160-f2ee7fc9f4df', 'langgraph_step': 2, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'mod

In [18]:
loaded_agui_events_1 = preview_saved_agui_events(thread_id=thread_id_1, limit=5)
print("First loaded event type:", loaded_agui_events_1[0]["type"] if loaded_agui_events_1 else None)

Loaded 392 AGUI events
[1] {
  "type": "CUSTOM",
  "timestamp": 1774090985911,
  "name": "PLAN_SNAPSHOT",
  "value": {
    "items": [
      {
        "content": "Conduct a brief overview search on LangGraph using available research tools.",
        "status": "in_progress",
        "metadata": null
      }
    ],
    "updated_at": 1774090985911,
    "metadata": {
      "namespace": null
    }
  }
}
[2] {
  "type": "CUSTOM",
  "timestamp": 1774090987289,
  "name": "TASK_SUBAGENT",
  "value": {
    "task_id": "call_gZfNbgqXewB6rmbmjUu23mYo",
    "subagent_type": "research-agent",
    "description": "Conduct research on LangGraph, summarizing its main features, usage, and applications in a concise report. The report should be clear and aimed at users unfamiliar with LangGraph."
  }
}
[3] {
  "type": "CUSTOM",
  "timestamp": 1774090987291,
  "name": "SUBAGENT_EVENT",
  "value": {
    "task_id": "bb42d6ea-5137-c15b-5c94-5358e9742e2b",
    "namespace": [
      "tools:bb42d6ea-5137-c15b-5c94-5

In [26]:
thread_id_2, output_path_2, agui_events_2 = await run_agui_handling_with_memory(
    "Give me a concise explanation of LangGraph, then summarize it in three bullets. Delegate 3 research subagents this time."
)

print("thread_id_2:", thread_id_2)
print("output_path_2:", output_path_2)
print("Different thread ids:", thread_id_1 != thread_id_2)
print("agui_events_2 count:", len(agui_events_2))


((), 'updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='Give me a concise explanation of LangGraph, then summarize it in three bullets. Delegate 3 research subagents this time.', additional_kwargs={}, response_metadata={}, id='d6548964-fa5d-47f2-ac39-61eddb94fe2a')])}})
-----\n\n
((), 'messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019d117c-602e-79a2-85fc-af09503658be', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 'thread_id': 'b559e8e4-2d06-4719-b160-f2ee7fc9f4df', 'langgraph_step': 2, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:39c2da49-7d62-65e9-2899-c74264c92c77', 'model': 'gpt-4o-mini', 'model_name': 'gpt-4o-mini', 'stream': False, '_type': 'openai-chat', 'checkpoint_ns': 'model:39c2da49-7d62-

In [27]:
loaded_agui_events_2 = preview_saved_agui_events(thread_id=thread_id_2, limit=5)
print("First loaded event type:", loaded_agui_events_2[0]["type"] if loaded_agui_events_2 else None)

Loaded 625 AGUI events
[1] {
  "type": "CUSTOM",
  "timestamp": 1774114861278,
  "name": "TASK_SUBAGENT",
  "value": {
    "task_id": "call_i93Gvd8oh8hUTx8BFlRxnQeN",
    "subagent_type": "research-agent",
    "description": "Research and provide a concise explanation of LangGraph, its features, and applications."
  }
}
[2] {
  "type": "CUSTOM",
  "timestamp": 1774114861278,
  "name": "TASK_SUBAGENT",
  "value": {
    "task_id": "call_LC31cwnD7zyUws1Q2fXw2h0J",
    "subagent_type": "research-agent",
    "description": "Summarize the key aspects of LangGraph in three bullet points, including its main use cases and advantages."
  }
}
[3] {
  "type": "CUSTOM",
  "timestamp": 1774114861278,
  "name": "TASK_SUBAGENT",
  "value": {
    "task_id": "call_NO6co5HevYSOuL4Y5bZKgWh0",
    "subagent_type": "research-agent",
    "description": "Research any recent developments or updates related to LangGraph."
  }
}
[4] {
  "type": "CUSTOM",
  "timestamp": 1774114861283,
  "name": "SUBAGENT_EVENT",


In [28]:
len(loaded_agui_events_2)

625

## Deep Agent - AGUI Full Handling (Files + HITL + Sub Agents)

This section stress-tests full deep-agent orchestration with:
- planning (`write_todos`)
- supervisor tools
- sub-agent delegation
- file handling tools
- HITL on both supervisor and sub-agent tools
- AG-UI event normalization for all streamed chunks


In [9]:
from pathlib import Path
from typing import Any, Dict, Optional
import json

from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command


FULL_WORKSPACE = Path("data/deepagents_full_handling")
FULL_WORKSPACE.mkdir(parents=True, exist_ok=True)


def _safe_workspace_path(filename: str) -> Path:
    candidate = (FULL_WORKSPACE / filename).resolve()
    root = FULL_WORKSPACE.resolve()
    if root not in candidate.parents and candidate != root:
        raise ValueError("File path must stay inside FULL_WORKSPACE")
    return candidate


@tool
def list_workspace_files() -> str:
    """List files currently stored in the notebook workspace."""
    files = sorted([p.name for p in FULL_WORKSPACE.glob("**/*") if p.is_file()])
    return json.dumps({"workspace": str(FULL_WORKSPACE), "files": files}, ensure_ascii=False)


@tool
def write_workspace_file(filename: str, content: str) -> str:
    """Create or overwrite a UTF-8 text file inside the notebook workspace."""
    path = _safe_workspace_path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    return f"WROTE {path.name} ({len(content)} chars)"


@tool
def append_workspace_file(filename: str, content: str) -> str:
    """Append UTF-8 text to a file inside the notebook workspace."""
    path = _safe_workspace_path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(content)
    return f"APPENDED {path.name} ({len(content)} chars)"


@tool
def read_workspace_file(filename: str) -> str:
    """Read a UTF-8 text file from the notebook workspace."""
    path = _safe_workspace_path(filename)
    if not path.exists():
        return f"MISSING FILE: {path.name}"
    return path.read_text(encoding="utf-8")


@tool
def deep_web_search(query: str) -> str:
    """Toy web search for deep-agent full handling tests."""
    return (
        "RESULT 1: (stub) LangGraph orchestrates agent workflows as state graphs.\n"
        "RESULT 2: (stub) Deep agents combine planning, delegation, and tool use.\n"
        f"QUERY: {query}"
    )


@tool
def deep_arxiv_search(query: str) -> str:
    """Toy arXiv search for deep-agent full handling tests."""
    return (
        "PAPER A: (stub) Tool-use and planning in agent systems.\n"
        "PAPER B: (stub) Multi-agent delegation and control loops.\n"
        f"QUERY: {query}"
    )


@tool
def policy_guard(action: str) -> str:
    """Policy check tool used to force supervisor-level HITL review when needed."""
    return f"Policy review completed for action: {action}"


full_research_subagent = {
    "name": "research-full",
    "description": "Research-focused sub-agent using web/arXiv tools.",
    "system_prompt": """
        You are the research-full sub-agent.
        Always do concise research and keep the response factual.
        Use both tools when the task asks for research.
    """,
    "tools": [deep_web_search, deep_arxiv_search],
    # Sub-agent HITL
    "interrupt_on": {
        "deep_arxiv_search": {"allowed_decisions": ["approve", "reject"]},
    },
}

full_files_subagent = {
    "name": "files-full",
    "description": "File-management sub-agent restricted to FULL_WORKSPACE.",
    "system_prompt": """
        You are the files-full sub-agent.
        Keep all file operations inside FULL_WORKSPACE.
        After file writes/appends, read back the file and report what changed.
    """,
    "tools": [list_workspace_files, write_workspace_file, append_workspace_file, read_workspace_file],
    # Sub-agent HITL
    "interrupt_on": {
        "append_workspace_file": True,
    },
}

full_supervisor_prompt = """
You are a deep-agent supervisor.

Rules:
1) For non-trivial requests, use write_todos to maintain a short plan.
2) Delegate research tasks to subagent 'research-full'.
3) Delegate filesystem tasks to subagent 'files-full'.
4) If user asks for potentially destructive/risky operational guidance,
    call policy_guard before final answer.
5) Keep final responses concise and practical.
"""


full_model = init_chat_model("openai:gpt-4o-mini")
full_checkpointer = MemorySaver()

full_agent = create_deep_agent(
    model=full_model,
    system_prompt=full_supervisor_prompt,
    tools=[policy_guard],
    subagents=[full_research_subagent, full_files_subagent],
    # Supervisor-level HITL
    interrupt_on={
        "policy_guard": True,
    },
    checkpointer=full_checkpointer,
)

In [10]:
def _decode_sse_payload(sse_event: bytes) -> Optional[Dict[str, Any]]:
    try:
        text = sse_event.decode("utf-8")
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("data:"):
            raw = line[len("data:"):].lstrip()
            try:
                payload = json.loads(raw)
            except Exception:
                return None
            return payload if isinstance(payload, dict) else None
    return None


def _sse_event_to_text(sse_event: Any) -> str:
    if isinstance(sse_event, bytes):
        try:
            return sse_event.decode("utf-8")
        except Exception:
            return repr(sse_event)
    return str(sse_event)


def _extract_interrupt_value_from_chunk(chunk: Any, normalizer: AGUIStreamNormalizer) -> Optional[Dict[str, Any]]:
    namespace, mode, payload, metadata = normalizer._unwrap_envelope(chunk)
    if mode != "updates" or not isinstance(payload, dict) or "__interrupt__" not in payload:
        return None

    raw = payload.get("__interrupt__")
    interrupt_obj = raw[0] if isinstance(raw, (tuple, list)) and raw else raw
    if interrupt_obj is None:
        return None

    value = getattr(interrupt_obj, "value", interrupt_obj)
    return value if isinstance(value, dict) else None


def _prompt_hitl_decisions(interrupt_value: Optional[Dict[str, Any]]) -> Optional[list[Dict[str, Any]]]:
    action_requests = []
    review_configs = []

    if isinstance(interrupt_value, dict):
        action_requests = interrupt_value.get("action_requests") or []
        review_configs = interrupt_value.get("review_configs") or []

    if not action_requests:
        decision = input("HITL interrupt detected. Decision [approve/reject/stop] (default: approve): ").strip().lower()
        if decision in {"", "approve"}:
            return [{"type": "approve"}]
        if decision == "reject":
            message = input("Reject message: ").strip() or "Rejected by user"
            return [{"type": "reject", "message": message}]
        return None

    allowed_by_action: Dict[str, list[str]] = {}
    for cfg in review_configs:
        if not isinstance(cfg, dict):
            continue
        action_name = cfg.get("action_name")
        allowed = cfg.get("allowed_decisions")
        if isinstance(action_name, str) and isinstance(allowed, list):
            allowed_by_action[action_name] = [str(x) for x in allowed]

    decisions: list[Dict[str, Any]] = []
    print("\nHITL interrupt decisions required:")

    for i, action in enumerate(action_requests, start=1):
        if not isinstance(action, dict):
            continue

        name = str(action.get("name", f"action_{i}"))
        args = action.get("args") if isinstance(action.get("args"), dict) else {}
        allowed = allowed_by_action.get(name, ["approve", "edit", "reject"])
        allowed_text = "/".join(allowed)

        print(f"- [{i}] tool={name} args={args}")
        raw_decision = input(f"  decision [{allowed_text}] (default: {allowed[0]}): ").strip().lower()
        decision_type = raw_decision if raw_decision in allowed else allowed[0]

        if decision_type == "approve":
            decisions.append({"type": "approve"})
            continue

        if decision_type == "reject":
            message = input("  reject message: ").strip() or f"Rejected tool call: {name}"
            decisions.append({"type": "reject", "message": message})
            continue

        if decision_type == "edit":
            edited_args_text = input(
                "  edited args JSON (empty = keep original args): "
            ).strip()
            edited_args = args
            if edited_args_text:
                try:
                    parsed = json.loads(edited_args_text)
                    if isinstance(parsed, dict):
                        edited_args = parsed
                    else:
                        print("  Invalid JSON object; keeping original args.")
                except Exception as exc:
                    print(f"  Invalid JSON ({exc}); keeping original args.")

            decisions.append(
                {
                    "type": "edit",
                    "edited_action": {
                        "name": name,
                        "args": edited_args,
                    },
                }
            )
            continue

        decisions.append({"type": "approve"})

    return decisions if decisions else None


def _format_event_brief(payload: Dict[str, Any]) -> str:
    ev_type = payload.get("type", "UNKNOWN")
    namespace = payload.get("namespace")

    if ev_type == "CUSTOM":
        return f"CUSTOM/{payload.get('name')} | ns={namespace}"

    if ev_type == "TOOL_CALL_START":
        return f"TOOL_CALL_START({payload.get('toolCallName')}) | ns={namespace}"

    if ev_type == "TOOL_CALL_END":
        return f"TOOL_CALL_END({payload.get('toolCallId')}) | ns={namespace}"

    return f"{ev_type} | ns={namespace}"


async def run_full_agui_normalized(
    user_text: str,
    *,
    thread_id: str,
    stream_mode: list[str] = ["updates", "messages"],
    subgraphs: bool = True,
    print_raw_chunks: bool = False,
    auto_resume_on_hitl: bool = True,
) -> None:
    config = {"configurable": {"thread_id": thread_id}}
    normalizer = AGUIStreamNormalizer(thread_id=thread_id)

    event_counts: Dict[str, int] = {}
    custom_counts: Dict[str, int] = {}

    pending_input: Any = {"messages": [{"role": "user", "content": user_text}]}
    run_index = 1

    while True:
        hitl_seen = False
        interrupt_value: Optional[Dict[str, Any]] = None

        print(f"\n=== Stream pass #{run_index} (thread_id={thread_id}) ===")

        async for chunk in full_agent.astream(
            pending_input,
            stream_mode=stream_mode,
            config=config,
            subgraphs=subgraphs,
        ):
            interrupt_from_chunk = _extract_interrupt_value_from_chunk(chunk, normalizer)
            if isinstance(interrupt_from_chunk, dict):
                interrupt_value = interrupt_from_chunk

            if print_raw_chunks:
                print("RAW CHUNK:", chunk)

            normalized = normalizer.handle_chunk(chunk)
            for sse_event in normalized:
                raw_text = _sse_event_to_text(sse_event)
                raw_hitl = '"type":"CUSTOM"' in raw_text and '"name":"HITL_INTERRUPT"' in raw_text

                # Keep this exact raw print for HITL trigger visibility.
                if raw_hitl:
                    print("AGUI EVENT (raw bytes):", raw_text)
                    hitl_seen = True

                payload = _decode_sse_payload(sse_event)
                if payload is None:
                    if not raw_hitl:
                        print("AGUI EVENT (raw bytes):", raw_text)
                    continue

                print(_format_event_brief(payload))

                ev_type = payload.get("type", "UNKNOWN")
                event_counts[ev_type] = event_counts.get(ev_type, 0) + 1
                if ev_type == "CUSTOM":
                    name = payload.get("name", "<missing>")
                    custom_counts[name] = custom_counts.get(name, 0) + 1

                    if name == "HITL_INTERRUPT":
                        hitl_seen = True
                        value_obj = payload.get("value") if isinstance(payload.get("value"), dict) else {}
                        intr_obj = value_obj.get("interrupt") if isinstance(value_obj, dict) else {}
                        intr_value = intr_obj.get("value") if isinstance(intr_obj, dict) else None
                        if isinstance(intr_value, dict):
                            interrupt_value = intr_value

        if hitl_seen and auto_resume_on_hitl:
            decisions = _prompt_hitl_decisions(interrupt_value)
            if not decisions:
                print("Stopping after HITL without resume decision.")
                break

            pending_input = Command(resume={"decisions": decisions})
            run_index += 1
            continue

        break

    print("\nEvent counts:", event_counts)
    print("Custom event counts:", custom_counts)


In [14]:
# Baseline full-scale run (planning + tools + subagents + files + HITL auto-resume)
full_thread_id = str(uuid.uuid4())

await run_full_agui_normalized(
    """
    Create a short plan.
    Delegate LangGraph research to research-full.
    Then delegate file work to files-full to create langgraph_notes.md with 3 concise bullets,
    read it back, and give a final 4-line summary.
    """,
    thread_id=full_thread_id,
    print_raw_chunks=True,
    auto_resume_on_hitl=True,
)

print("full_thread_id:", full_thread_id)


=== Stream pass #1 (thread_id=113f9848-8e33-4cc1-bc53-37cc6538516b) ===
RAW CHUNK: ((), 'updates', {'PatchToolCallsMiddleware.before_agent': {'messages': Overwrite(value=[HumanMessage(content='\n    Create a short plan.\n    Delegate LangGraph research to research-full.\n    Then delegate file work to files-full to create langgraph_notes.md with 3 concise bullets,\n    read it back, and give a final 4-line summary.\n    ', additional_kwargs={}, response_metadata={}, id='3a2a0e50-e7b8-4e3c-9a9b-f0506584f336')])}})
RAW CHUNK: ((), 'updates', {'SummarizationMiddleware.before_model': None})
RAW CHUNK: ((), 'messages', (AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--da182971-e1f0-420e-818a-d5fb0d6ea451', tool_calls=[{'name': 'write_todos', 'args': {}, 'id': 'call_DJ4aXaeb0oKuWx1PDXB3HhM7', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'write_todos', 'args': '', 'id': 'call_DJ4aXaeb0oKuWx1PDXB3HhM7', 'index': 0, 'type': 'tool

In [12]:
# Sub-agent HITL run (append_workspace_file interrupt + interactive resume)
await run_full_agui_normalized(
    """
    Continue from previous context.
    Ask files-full to append one more bullet to langgraph_notes.md and read the file again.
    """,
    thread_id=full_thread_id,
    auto_resume_on_hitl=True,
)


=== Stream pass #1 (thread_id=4da9ec68-a0b9-4937-95e3-87c18ca5ec54) ===
AGUI EVENT (raw bytes): data: {"type":"CUSTOM","timestamp":1773419578440,"name":"PLAN_SNAPSHOT","value":{"items":[{"content":"Delegate LangGraph research to 'research-full'.","status":"in_progress","metadata":null},{"content":"Create 'langgraph_notes.md' with 3 concise bullets.","status":"pending","metadata":null},{"content":"Read 'langgraph_notes.md' back.","status":"pending","metadata":null},{"content":"Provide a final 4-line summary.","status":"pending","metadata":null}],"updated_at":1773419578440,"metadata":{"namespace":null}}}


AGUI EVENT (raw bytes): data: {"type":"CUSTOM","timestamp":1773419578440,"name":"TASK_SUBAGENT","value":{"task_id":"call_tzAb7tHRcf8VyYHbj1pVVvWp","subagent_type":"research-full","description":"Conduct research on LangGraph, focusing on its features, benefits, and use cases. Provide a concise summary of the findings in bullet format for inclusion in a notes file."}}


AGUI EVENT (raw 

In [13]:
# Supervisor HITL run (policy_guard interrupt + interactive resume)
await run_full_agui_normalized(
    """
    Continue from previous context.
    Give me operational guidance for a destructive action that could wipe all project files.
    Keep it short.
    """,
    thread_id=full_thread_id,
    auto_resume_on_hitl=True,
)

print("Workspace files now:", [p.name for p in FULL_WORKSPACE.glob("**/*") if p.is_file()])
notes_path = FULL_WORKSPACE / "langgraph_notes.md"
if notes_path.exists():
    print("\nlanggraph_notes.md\n------------------")
    print(notes_path.read_text(encoding="utf-8"))



=== Stream pass #1 (thread_id=4da9ec68-a0b9-4937-95e3-87c18ca5ec54) ===
AGUI EVENT (raw bytes): data: {"type":"CUSTOM","timestamp":1773419671372,"name":"PLAN_SNAPSHOT","value":{"items":[{"content":"Delegate LangGraph research to 'research-full'.","status":"in_progress","metadata":null},{"content":"Create 'langgraph_notes.md' with 3 concise bullets.","status":"pending","metadata":null},{"content":"Read 'langgraph_notes.md' back.","status":"pending","metadata":null},{"content":"Provide a final 4-line summary.","status":"pending","metadata":null}],"updated_at":1773419671372,"metadata":{"namespace":null}}}


AGUI EVENT (raw bytes): data: {"type":"CUSTOM","timestamp":1773419671372,"name":"TASK_SUBAGENT","value":{"task_id":"call_tzAb7tHRcf8VyYHbj1pVVvWp","subagent_type":"research-full","description":"Conduct research on LangGraph, focusing on its features, benefits, and use cases. Provide a concise summary of the findings in bullet format for inclusion in a notes file."}}


AGUI EVENT (raw 